# सबक 08 - मल्टी-एजेंट डिज़ाइन पैटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मल्टी-एजेंट सिस्टम क्यों?

वास्तविक दुनिया के कार्य जैसे यात्रा योजना में कई प्रकार की विशेषज्ञता आवश्यक होती है — लॉजिस्टिक्स, स्थानीय ज्ञान, बजटिंग, और अधिक। एक अकेला एजेंट सब कुछ संभालने की कोशिश करता है तो वह जल्दी ही असंगठित हो जाता है।

मल्टी-एजेंट सिस्टम इसे **विशेषीकरण** के माध्यम से हल करते हैं: प्रत्येक एजेंट एक विशेषज्ञता के क्षेत्र पर केंद्रित होता है, जो सामान्यज्ञ की तुलना में बेहतर गुणवत्ता वाले परिणाम प्रदान करता है। वे **विस्तारशीलता** भी बढ़ाते हैं — आप नए एजेंट जोड़ सकते हैं (जैसे, एक फ्लाइट विशेषज्ञ, एक रेस्तरां आलोचक) बिना मौजूदा कार्यप्रवाह को फिर से लिखा। एजेंट एक सुव्यवस्थित पाइपलाइन के माध्यम से आपस में जुड़ते हैं, और संदर्भ एक से दूसरे को पारित करते हैं।


## विशेषज्ञ एजेंट बनाना


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## एक अनुक्रमिक वर्कफ़्लो बनाना

`WorkflowBuilder` आपको एजेंट्स को एक निर्देशित ग्राफ़ में जोड़ने देता है। यहां हम एक सरल दो-चरणीय पाइपलाइन बनाते हैं: **TravelPlanner** यात्रा कार्यक्रम तैयार करता है, फिर **TravelConcierge** उसे समीक्षा करके बेहतर बनाता है।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ़्लो में अधिक एजेंट जोड़ना

मल्टी-एजेंट पैटर्न का सबसे बड़ा लाभ यह है कि इसे बढ़ाना कितना आसान है। नीचे हमने एक **BudgetReviewer** एजेंट जोड़ा है जो योजना को यात्री के बजट के खिलाफ जांचता है, उन वस्तुओं को चिह्नित करता है जो लागत सीमा से ऊपर जा सकती हैं, और पैसे बचाने वाले विकल्प सुझाता है। वर्कफ़्लो अब तीन एजेंटों को क्रम में चलाता है:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **विशेषीकृत एजेंट बनाएं** — प्रत्येक एक केंद्रित भूमिका के साथ (योजना बनाना, कंसीयर्ज, बजट समीक्षा)।
2. **एजेंट्स को एक क्रमिक वर्कफ़्लो में जोड़ें** `WorkflowBuilder` और `add_edge` का उपयोग करके।
3. **मल्टी-एजेंट पाइपलाइन से आउटपुट स्ट्रीम करें**, यह ट्रैक करते हुए कि कौन सा एजेंट बोल रहा है।
4. **वर्कफ़्लो का विस्तार करें** बिना मौजूदा एजेंट्स को बदले, चेन में नए एजेंट जोड़ कर।

मल्टी-एजेंट डिज़ाइन पैटर्न प्रत्येक एजेंट को सरल रखता है जबकि अधिक समृद्ध, विस्तृत समीक्षा किए गए परिणाम प्रदान करता है जो कोई भी एकल एजेंट अकेले हासिल नहीं कर सकता।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**डिस्क्लेमर**:
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनूदित किया गया है। हम सटीकता के लिए प्रयासरत हैं, लेकिन कृपया ध्यान रखें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल भाषा में दस्तावेज़ को आधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
